# musegen: symbolic music generation in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edwin580/music-generator/blob/main/notebooks/musegen_colab.ipynb)

This notebook drives the [`musegen`](https://github.com/edwin580/music-generator) package on a Colab GPU:
train a Transformer (or the LSTM baselines) on MIDI, generate new melodies in a chosen key, listen to them,
and evaluate them against held-out music.

> **Runtime → Change runtime type → GPU** makes training roughly 20× faster.

In [ ]:
# Install the package, plus FluidSynth so rendered audio uses a real piano SoundFont
!pip -q install git+https://github.com/edwin580/music-generator.git pyfluidsynth
!apt-get -qq install -y fluidsynth > /dev/null

## 1. Get some MIDI

Pick one:
* **MIDIWorld sample set**, the files used by the original COGS-319 notebook;
* **synthetic corpus**, procedurally composed melodies (works offline, good for a quick check);
* **your own files**: upload `.mid` files to `data/midi/`.

In [ ]:
DATASET = "midiworld"  # "midiworld" | "synthetic" | "own"

if DATASET == "midiworld":
    !musegen download --dest data/midi
    MIDI_DIR = "data/midi"
elif DATASET == "synthetic":
    !musegen synth-data --dest data/synthetic --n 256
    MIDI_DIR = "data/synthetic"
else:
    MIDI_DIR = "data/midi"

## 2. Train

Presets ship with the package (`src/musegen/configs/`). `transformer` is the recommended model; `lstm` and `pianoroll_lstm`
(the original notebook's model, fixed) are baselines. Any setting can be overridden with `--set`.

In [ ]:
ARCH = "transformer"  # "transformer" | "lstm" | "pianoroll_lstm"
!musegen train -c {ARCH} --midi-dir {MIDI_DIR} --output-dir runs/{ARCH} --set train.epochs=40

In [ ]:
from IPython.display import Image, display
display(Image("runs/" + ARCH + "/training_history.png"))

## 3. Generate

* `--key` restricts generated pitches to a scale (`"auto"` detects the key of the prompt);
* `--prompt` continues the opening bars of any MIDI file;
* `--temperature` / `--top-p` trade coherence for surprise.

In [ ]:
!musegen generate -m runs/{ARCH}/best.pt -o output/song.mid --bars 16 --key "A minor" --temperature 0.9 --top-p 0.95 --seed 7 --plot --wav
display(Image("output/song.png"))
from IPython.display import Audio
Audio("output/song.wav")

In [ ]:
# Continue an existing piece: 4 bars of prompt, 12 bars generated, in the prompt's own key
import glob
prompt = sorted(glob.glob(MIDI_DIR + "/*.mid"))[0]
!musegen generate -m runs/{ARCH}/best.pt -o output/continuation.mid --prompt "{prompt}" --prompt-bars 4 --bars 12 --key auto --plot --wav
display(Image("output/continuation.png"))
Audio("output/continuation.wav")

## 4. Evaluate

Reports held-out perplexity and compares generated continuations with the real continuations of held-out
pieces on pitch, interval, rhythm, key and repetition statistics.

In [ ]:
!musegen evaluate -m runs/{ARCH}/best.pt --num-samples 32
display(Image("runs/" + ARCH + "/eval/distributions.png"))
display(Image("runs/" + ARCH + "/eval/sample_00_piano_roll.png"))

In [ ]:
from google.colab import files
files.download("output/song.mid")